In [1]:
# ============================================================
# 训练集、验证集、测试集均可手动指定版本
# 适配不等距 EPIC 时间序列版本：保留 TCN
# 你以后只需要修改最下面的 USER_CONFIG
# ============================================================

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import shap
import json
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score


# ============================================================
# 0. 站点划分工具函数
# ============================================================

def check_site_overlap(train_sites, val_sites, test_sites):
    """
    检查训练集、验证集、测试集是否有重复站点，避免数据泄漏。
    """
    overlap_train_val = set(train_sites) & set(val_sites)
    overlap_train_test = set(train_sites) & set(test_sites)
    overlap_val_test = set(val_sites) & set(test_sites)

    if overlap_train_val:
        raise ValueError(f"训练集和验证集有重复站点: {sorted(overlap_train_val)}")

    if overlap_train_test:
        raise ValueError(f"训练集和测试集有重复站点: {sorted(overlap_train_test)}")

    if overlap_val_test:
        raise ValueError(f"验证集和测试集有重复站点: {sorted(overlap_val_test)}")


def split_files_by_manual_sites(all_files, train_sites, val_sites, test_sites):
    """
    根据手动指定的 train_sites / val_sites / test_sites 划分 CSV 文件。

    匹配逻辑：
    文件名去掉 .csv 后，满足：
    1. stem == site
    或
    2. stem.startswith(site + "_")

    例如：
    site = AU-Cpr_WSA

    可以匹配：
    AU-Cpr_WSA.csv
    AU-Cpr_WSA_2016-2020.csv
    AU-Cpr_WSA_HH.csv
    """
    train_files = []
    val_files = []
    test_files = []

    matched_train_sites = set()
    matched_val_sites = set()
    matched_test_sites = set()

    ignored_files = []

    for f in all_files:
        fname = os.path.basename(f)
        stem = os.path.splitext(fname)[0]

        matched = False

        # 优先匹配测试集
        for site in test_sites:
            if stem == site or stem.startswith(site + "_"):
                test_files.append(f)
                matched_test_sites.add(site)
                matched = True
                break

        if matched:
            continue

        # 再匹配验证集
        for site in val_sites:
            if stem == site or stem.startswith(site + "_"):
                val_files.append(f)
                matched_val_sites.add(site)
                matched = True
                break

        if matched:
            continue

        # 最后匹配训练集
        for site in train_sites:
            if stem == site or stem.startswith(site + "_"):
                train_files.append(f)
                matched_train_sites.add(site)
                matched = True
                break

        if not matched:
            ignored_files.append(f)

    print("\n📁 文件划分完成")
    print(f"   文件夹内 CSV 总数: {len(all_files)}")
    print(f"   训练集文件数: {len(train_files)}")
    print(f"   验证集文件数: {len(val_files)}")
    print(f"   测试集文件数: {len(test_files)}")
    print(f"   未使用 CSV 文件数: {len(ignored_files)}")

    missing_train_sites = sorted(set(train_sites) - matched_train_sites)
    missing_val_sites = sorted(set(val_sites) - matched_val_sites)
    missing_test_sites = sorted(set(test_sites) - matched_test_sites)

    if missing_train_sites:
        print("\n⚠️ 以下训练集站点没有在文件夹中匹配到 CSV 文件:")
        for s in missing_train_sites:
            print(f"   - {s}")

    if missing_val_sites:
        print("\n⚠️ 以下验证集站点没有在文件夹中匹配到 CSV 文件:")
        for s in missing_val_sites:
            print(f"   - {s}")

    if missing_test_sites:
        print("\n⚠️ 以下测试集站点没有在文件夹中匹配到 CSV 文件:")
        for s in missing_test_sites:
            print(f"   - {s}")

    if len(train_files) == 0:
        raise ValueError("训练集文件数为 0，请检查 train_sites 和 CSV 文件名是否一致。")

    if len(val_files) == 0:
        raise ValueError("验证集文件数为 0，请检查 val_sites 和 CSV 文件名是否一致。")

    if len(test_files) == 0:
        raise ValueError("测试集文件数为 0，请检查 test_sites 和 CSV 文件名是否一致。")

    return train_files, val_files, test_files


# ============================================================
# 1. 数据集定义：适配不等距 EPIC 时间序列
# ============================================================

class TimeAwareMultiStationDataset(Dataset):
    def __init__(
        self,
        filepaths,
        seq_len=96,
        target_col='GPP_DT_VUT_REF',
        time_col='date',
        forcing_cols=None,
        state_cols=None,
        static_cols=None,
        lc_col='Veg_ID',
        feat_mean_f=None,
        feat_std_f=None,
        feat_mean_s=None,
        feat_std_s=None,
        static_mean=None,
        static_std=None,
        target_mean=None,
        target_std=None,
        split_type='train',

        # =========================
        # 新增：不等距时间序列控制参数
        # =========================
        max_gap_hours=6,
        max_span_hours=168,
        dt_clip_hours=240
    ):
        """
        关键修改说明：

        原代码要求窗口满足：
            actual_duration == mode_step * (seq_len - 1)

        这等价于假设时间序列严格等距。

        现在改为：
            1. 不要求严格等距；
            2. 只要窗口内部相邻时间间隔不超过 max_gap_hours；
            3. 整个窗口总跨度不超过 max_span_hours；
            4. 显式加入 dt_prev_hours 和 age_to_target_hours 时间特征。

        time_x 最终维度：
            6 维：
            [sin(hour), cos(hour), sin(dayofyear), cos(dayofyear),
             log1p(dt_prev_hours), log1p(age_to_target_hours)]
        """
        self.seq_len = seq_len
        self.samples = []

        self.max_gap_hours = max_gap_hours
        self.max_span_hours = max_span_hours
        self.dt_clip_hours = dt_clip_hours

        self.station_forcing = []
        self.station_state = []
        self.station_time_features = []
        self.station_targets = []
        self.station_dates = []

        self.station_static = []
        self.station_lc = []

        if forcing_cols is None:
            forcing_cols = []

        if state_cols is None:
            state_cols = []

        if static_cols is None:
            static_cols = []

        for filepath in filepaths:
            fname = os.path.basename(filepath)

            try:
                data = pd.read_csv(filepath)
            except Exception as e:
                print(f"⚠️ 文件 {fname} 读取失败，跳过。原因: {e}")
                continue

            if time_col not in data.columns:
                print(f"⚠️ 在文件 {fname} 中找不到时间列 '{time_col}'，跳过。")
                continue

            required_input_cols = forcing_cols + state_cols + static_cols + [lc_col]
            missing_input_cols = [c for c in required_input_cols if c not in data.columns]

            if missing_input_cols:
                print(f"⚠️ 文件 {fname} 缺少以下输入变量，跳过:")
                for c in missing_input_cols:
                    print(f"   - {c}")
                continue

            if target_col not in data.columns:
                print(f"⚠️ 文件 {fname} 中找不到目标列 '{target_col}'，跳过。")
                continue

            # 将 -9999 等无效值转为 NaN
            data = data.replace([-9999, -9999.0, -999], np.nan)

            # 解析时间
            data[time_col] = pd.to_datetime(data[time_col], errors='coerce')
            data = data.dropna(subset=[time_col]).reset_index(drop=True)

            if len(data) == 0:
                print(f"⚠️ 文件 {fname} 时间列全部无效，跳过。")
                continue

            # 按时间排序
            data = data.sort_values(by=time_col).reset_index(drop=True)

            # 删除重复时间
            if data[time_col].duplicated().any():
                dup_count = int(data[time_col].duplicated().sum())
                print(f"⚠️ 文件 {fname} 存在 {dup_count} 条重复时间，已保留最后一条记录。")
                data = data.drop_duplicates(subset=[time_col], keep='last').reset_index(drop=True)

            # 清理输入变量和目标变量的缺失
            cols_to_clean = forcing_cols + state_cols + static_cols + [lc_col, target_col]
            data = data.dropna(subset=cols_to_clean).reset_index(drop=True)

            if len(data) < seq_len:
                print(f"⚠️ 文件 {fname} 清洗后长度 {len(data)} 小于 seq_len={seq_len}，跳过。")
                continue

            # 再次排序，确保清洗后仍按时间递增
            data = data.sort_values(by=time_col).reset_index(drop=True)

            dates = data[time_col]
            dates_array = dates.values.astype('datetime64[ns]')
            self.station_dates.append(dates_array)

            # ====================================================
            # 时间特征 1：周期时间特征
            # ====================================================
            hours = dates.dt.hour.values
            days = dates.dt.dayofyear.values

            # ====================================================
            # 时间特征 2：当前观测距离上一次观测的时间间隔
            # 不等距序列的关键新增变量
            # ====================================================
            dt_prev_hours = dates.diff().dt.total_seconds().div(3600).values
            dt_prev_hours[0] = 0.0

            dt_prev_hours = np.nan_to_num(
                dt_prev_hours,
                nan=0.0,
                posinf=dt_clip_hours,
                neginf=0.0
            )

            dt_prev_hours = np.clip(
                dt_prev_hours,
                0.0,
                dt_clip_hours
            )

            # 这里先构造 5 维时间基础特征
            # 第 6 维 age_to_target_hours 需要在 __getitem__ 中根据窗口目标时刻动态生成
            time_base_feats = np.column_stack([
                np.sin(2 * np.pi * hours / 24.0),
                np.cos(2 * np.pi * hours / 24.0),
                np.sin(2 * np.pi * days / 365.25),
                np.cos(2 * np.pi * days / 365.25),
                np.log1p(dt_prev_hours)
            ]).astype(np.float32)

            forcing_data = data[forcing_cols].values.astype(np.float32)
            state_data = data[state_cols].values.astype(np.float32)
            static_data = data[static_cols].values.astype(np.float32)
            lc_data = data[lc_col].values.astype(np.int64)
            targets = data[target_col].values.astype(np.float32)

            self.station_forcing.append(forcing_data)
            self.station_state.append(state_data)
            self.station_time_features.append(time_base_feats)
            self.station_targets.append(targets)
            self.station_static.append(static_data)
            self.station_lc.append(lc_data)

        if not self.station_forcing:
            raise ValueError(f"加载 {split_type} 数据失败，可能所有数据均被清洗掉、缺列或长度不足。")

        # ========================================================
        # 标准化参数：只从训练集计算，验证/测试集复用训练集参数
        # ========================================================

        all_forcing_concat = np.vstack(self.station_forcing)
        all_state_concat = np.vstack(self.station_state)
        all_static_concat = np.vstack(self.station_static)
        all_targets_concat = np.concatenate(self.station_targets)

        self.feat_mean_f = np.mean(all_forcing_concat, axis=0) if feat_mean_f is None else feat_mean_f
        self.feat_std_f = np.std(all_forcing_concat, axis=0) if feat_std_f is None else feat_std_f
        self.feat_std_f[self.feat_std_f == 0] = 1e-8

        self.feat_mean_s = np.mean(all_state_concat, axis=0) if feat_mean_s is None else feat_mean_s
        self.feat_std_s = np.std(all_state_concat, axis=0) if feat_std_s is None else feat_std_s
        self.feat_std_s[self.feat_std_s == 0] = 1e-8

        self.static_mean = np.mean(all_static_concat, axis=0) if static_mean is None else static_mean
        self.static_std = np.std(all_static_concat, axis=0) if static_std is None else static_std
        self.static_std[self.static_std == 0] = 1e-8

        self.target_mean = np.mean(all_targets_concat) if target_mean is None else target_mean
        self.target_std = np.std(all_targets_concat) if target_std is None else target_std

        if self.target_std == 0:
            self.target_std = 1e-8

        # ========================================================
        # 应用标准化
        # ========================================================

        for i in range(len(self.station_forcing)):
            self.station_forcing[i] = (self.station_forcing[i] - self.feat_mean_f) / self.feat_std_f
            self.station_state[i] = (self.station_state[i] - self.feat_mean_s) / self.feat_std_s
            self.station_static[i] = (self.station_static[i] - self.static_mean) / self.static_std
            self.station_targets[i] = (self.station_targets[i] - self.target_mean) / self.target_std

        # ========================================================
        # 构造样本窗口：不再要求严格等距
        # ========================================================

        total_candidate_windows = 0
        total_valid_windows = 0
        total_rejected_by_gap = 0
        total_rejected_by_span = 0

        for i in range(len(self.station_forcing)):
            dates_series = pd.Series(pd.to_datetime(self.station_dates[i]))

            num_samples = len(self.station_forcing[i]) - self.seq_len + 1
            if num_samples <= 0:
                continue

            total_candidate_windows += num_samples

            for j in range(num_samples):
                window_dates = dates_series.iloc[j: j + self.seq_len]

                window_diffs_hours = (
                    window_dates
                    .diff()
                    .dt.total_seconds()
                    .div(3600)
                    .dropna()
                )

                if len(window_diffs_hours) == 0:
                    continue

                # 防止重复时间或异常倒序时间进入窗口
                if (window_diffs_hours <= 0).any():
                    total_rejected_by_gap += 1
                    continue

                # 1. 不允许窗口跨过过大的时间断点
                if max_gap_hours is not None:
                    if (window_diffs_hours > max_gap_hours).any():
                        total_rejected_by_gap += 1
                        continue

                # 2. 不允许一个窗口总时间跨度过大
                if max_span_hours is not None:
                    span_hours = (
                        window_dates.iloc[-1] - window_dates.iloc[0]
                    ).total_seconds() / 3600.0

                    if span_hours > max_span_hours:
                        total_rejected_by_span += 1
                        continue

                self.samples.append((i, j))
                total_valid_windows += 1

        print(f"\n📌 {split_type} 数据集窗口构造完成")
        print(f"   候选窗口数: {total_candidate_windows}")
        print(f"   有效窗口数: {total_valid_windows}")
        print(f"   因时间断点过大/重复时间剔除: {total_rejected_by_gap}")
        print(f"   因窗口总跨度过大剔除: {total_rejected_by_span}")
        print(f"   max_gap_hours: {max_gap_hours}")
        print(f"   max_span_hours: {max_span_hours}")

        if len(self.samples) == 0:
            raise ValueError(
                f"{split_type} 数据集中没有形成任何有效序列样本。"
                f"请检查 seq_len={seq_len}, max_gap_hours={max_gap_hours}, "
                f"max_span_hours={max_span_hours} 是否过于严格。"
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        station_idx, start_idx = self.samples[idx]

        end_idx = start_idx + self.seq_len

        x_forcing = self.station_forcing[station_idx][start_idx:end_idx]
        x_state = self.station_state[station_idx][start_idx:end_idx]

        # 前 5 维时间基础特征
        time_base_x = self.station_time_features[station_idx][start_idx:end_idx]

        # ========================================================
        # 第 6 维时间特征：每个历史步距离目标时刻的时间差
        # age_to_target_hours
        # ========================================================

        window_dates = self.station_dates[station_idx][start_idx:end_idx]
        target_date = window_dates[-1]

        age_to_target_hours = (
            (target_date - window_dates) / np.timedelta64(1, 'h')
        ).astype(np.float32)

        age_to_target_hours = np.nan_to_num(
            age_to_target_hours,
            nan=0.0,
            posinf=self.dt_clip_hours,
            neginf=0.0
        )

        age_to_target_hours = np.clip(
            age_to_target_hours,
            0.0,
            self.dt_clip_hours
        )

        age_to_target_feat = np.log1p(age_to_target_hours).reshape(-1, 1)

        # 最终 time_x 为 6 维
        time_x = np.concatenate(
            [time_base_x, age_to_target_feat],
            axis=1
        ).astype(np.float32)

        y = self.station_targets[station_idx][end_idx - 1]

        x_static = self.station_static[station_idx][start_idx:end_idx]
        x_lc = self.station_lc[station_idx][start_idx:end_idx]

        return (
            torch.tensor(x_forcing, dtype=torch.float32),
            torch.tensor(x_state, dtype=torch.float32),
            torch.tensor(time_x, dtype=torch.float32),
            torch.tensor(x_static, dtype=torch.float32),
            torch.tensor(x_lc, dtype=torch.long),
            torch.tensor(y, dtype=torch.float32),
            str(target_date)
        )


# ============================================================
# 2. TCN 基础模块
# ============================================================

class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()

        self.conv1 = nn.Conv1d(
            n_inputs,
            n_outputs,
            kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation
        )

        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            n_outputs,
            n_outputs,
            kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation
        )

        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(
            self.conv1,
            self.chomp1,
            self.relu1,
            self.dropout1,
            self.conv2,
            self.chomp2,
            self.relu2,
            self.dropout2
        )

        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)


class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=3, dropout=0.2):
        super(TemporalConvNet, self).__init__()

        layers = []
        num_levels = len(num_channels)

        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i - 1]
            out_channels = num_channels[i]

            layers += [
                TemporalBlock(
                    in_channels,
                    out_channels,
                    kernel_size,
                    stride=1,
                    dilation=dilation_size,
                    padding=(kernel_size - 1) * dilation_size,
                    dropout=dropout
                )
            ]

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# ============================================================
# 3. TCN + Transformer + Cross Attention 模型
#    保留 TCN，只把 time_projector 从 4 维改为 6 维
# ============================================================

class TCN_Transformer_CrossAttention(nn.Module):
    def __init__(
        self,
        num_forcing_features,
        num_state_features,
        seq_len,
        num_static=2,
        num_lc_classes=13,
        lc_embed_dim=8,
        time_feature_dim=6,
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.1
    ):
        super(TCN_Transformer_CrossAttention, self).__init__()

        self.seq_len = seq_len
        self.time_feature_dim = time_feature_dim

        self.tcn = TemporalConvNet(
            num_inputs=num_forcing_features,
            num_channels=[d_model] * 6,
            kernel_size=3,
            dropout=dropout
        )

        self.lc_embedding = nn.Embedding(
            num_embeddings=num_lc_classes,
            embedding_dim=lc_embed_dim
        )

        combined_state_dim = num_state_features + num_static + lc_embed_dim

        self.state_linear = nn.Linear(combined_state_dim, d_model)

        # 原来是 nn.Linear(4, d_model)
        # 现在改成 nn.Linear(6, d_model)
        self.time_projector = nn.Linear(time_feature_dim, d_model)

        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layers,
            num_layers
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True
        )

        self.regressor = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4),
            nn.ReLU(),
            nn.Linear(d_model // 4, 1)
        )

    def forward(self, x_forcing, x_state, time_x, x_static, x_lc):
        # forcing 分支：TCN
        # x_forcing: [batch, seq_len, num_forcing_features]
        x_tcn_in = x_forcing.transpose(1, 2)
        f_tcn = self.tcn(x_tcn_in)
        f_met_memory = f_tcn.transpose(1, 2)

        # state + static + land cover 分支
        lc_emb = self.lc_embedding(x_lc)
        combined_state = torch.cat([x_state, x_static, lc_emb], dim=-1)

        x_s_emb = self.state_linear(combined_state)
        time_emb = self.time_projector(time_x)

        x_state_combined = x_s_emb + time_emb
        f_state_global = self.transformer_encoder(x_state_combined)

        # Cross Attention:
        # query 来自状态/遥感分支
        # key/value 来自 forcing TCN 分支
        fused_features, _ = self.cross_attention(
            query=f_state_global,
            key=f_met_memory,
            value=f_met_memory
        )

        # 只预测窗口最后一个时刻的 GPP
        last_step_features = fused_features[:, -1, :]
        out = self.regressor(last_step_features)

        return out.squeeze(-1)


# ============================================================
# 4. SHAP 分析函数
# ============================================================

def perform_shap_analysis(model, dataloader, device, output_img_folder, forcing_cols, state_cols):
    print("\n🔍 开始执行 SHAP 变量重要性分析...")
    model.eval()

    shap_loader = DataLoader(
        dataloader.dataset,
        batch_size=4000,
        shuffle=True
    )

    batch = next(iter(shap_loader))

    batch_forcing = batch[0].to(device)
    batch_state = batch[1].to(device)
    batch_time = batch[2].to(device)
    batch_static = batch[3].to(device)
    batch_lc = batch[4].to(device)

    bg_size = 1000
    test_size = 1000

    if batch_forcing.size(0) < (bg_size + test_size):
        print("⚠️ Batch size 太小，跳过 SHAP 分析。")
        return

    bg_f = batch_forcing[:bg_size]
    bg_s = batch_state[:bg_size]

    test_f = batch_forcing[bg_size:bg_size + test_size]
    test_s = batch_state[bg_size:bg_size + test_size]

    class SHAPWrapper(nn.Module):
        def __init__(self, base_model, t_ref, st_ref, lc_ref):
            super().__init__()
            self.base_model = base_model
            self.t_ref = t_ref[:1]
            self.st_ref = st_ref[:1]
            self.lc_ref = lc_ref[:1]

        def forward(self, x_f, x_s):
            b_size = x_f.size(0)

            t_x = self.t_ref.expand(b_size, -1, -1)
            x_st = self.st_ref.expand(b_size, -1, -1)
            x_l = self.lc_ref.expand(b_size, -1)

            out = self.base_model(
                x_f,
                x_s,
                t_x,
                x_st,
                x_l
            )

            return out.unsqueeze(-1)

    wrapper_model = SHAPWrapper(
        model,
        batch_time,
        batch_static,
        batch_lc
    ).to(device)

    wrapper_model.eval()

    explainer = shap.GradientExplainer(
        wrapper_model,
        [bg_f, bg_s]
    )

    shap_values = explainer.shap_values(
        [test_f, test_s]
    )

    if isinstance(shap_values, list) and len(shap_values) == 1 and isinstance(shap_values[0], list):
        shap_values = shap_values[0]

    shap_forcing = np.array(shap_values[0])
    shap_state = np.array(shap_values[1])

    shap_forcing_2d = shap_forcing.reshape(-1, len(forcing_cols))
    shap_state_2d = shap_state.reshape(-1, len(state_cols))

    test_f_2d = test_f.cpu().numpy().reshape(-1, len(forcing_cols))
    test_s_2d = test_s.cpu().numpy().reshape(-1, len(state_cols))

    shap_combined = np.concatenate(
        [shap_forcing_2d, shap_state_2d],
        axis=1
    )

    features_combined = np.concatenate(
        [test_f_2d, test_s_2d],
        axis=1
    )

    feature_names = forcing_cols + state_cols

    plt.figure(figsize=(12, 10))

    shap.summary_plot(
        shap_combined,
        features_combined,
        feature_names=feature_names,
        show=False
    )

    plt.title(
        "SHAP Summary: Global Feature Impact on GPP (Time-Flattened)",
        fontname='Arial',
        fontsize=14
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(output_img_folder, "SHAP_Summary_Plot.png"),
        dpi=300
    )

    plt.close()

    plt.figure(figsize=(12, 10))

    shap.summary_plot(
        shap_combined,
        features_combined,
        feature_names=feature_names,
        plot_type="bar",
        show=False
    )

    plt.title(
        "SHAP Global Feature Importance (Magnitude)",
        fontname='Arial',
        fontsize=14
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(output_img_folder, "SHAP_Bar_Plot.png"),
        dpi=300
    )

    plt.close()

    print(f"✅ SHAP 生成完成，保存至: {output_img_folder}")


# ============================================================
# 5. 保存训练参数
# ============================================================

def save_training_artifacts(
    output_img_folder,
    seq_len,
    time_col,
    target_col,
    forcing_cols,
    state_cols,
    static_cols,
    lc_col,
    num_lc_classes,
    lc_embed_dim,
    time_feature_dim,
    max_gap_hours,
    max_span_hours,
    dt_clip_hours,
    train_files,
    val_files,
    test_files,
    train_sites,
    val_sites,
    test_sites,
    f_mean_f,
    f_std_f,
    f_mean_s,
    f_std_s,
    s_mean,
    s_std,
    t_mean,
    t_std
):
    os.makedirs(output_img_folder, exist_ok=True)

    scaler_npz_path = os.path.join(output_img_folder, "global_scalers.npz")
    config_json_path = os.path.join(output_img_folder, "model_input_config.json")

    np.savez(
        scaler_npz_path,
        feat_mean_f=np.asarray(f_mean_f, dtype=np.float32),
        feat_std_f=np.asarray(f_std_f, dtype=np.float32),
        feat_mean_s=np.asarray(f_mean_s, dtype=np.float32),
        feat_std_s=np.asarray(f_std_s, dtype=np.float32),
        static_mean=np.asarray(s_mean, dtype=np.float32),
        static_std=np.asarray(s_std, dtype=np.float32),
        target_mean=np.asarray(t_mean, dtype=np.float32),
        target_std=np.asarray(t_std, dtype=np.float32),
        forcing_cols=np.asarray(forcing_cols, dtype=object),
        state_cols=np.asarray(state_cols, dtype=object),
        static_cols=np.asarray(static_cols, dtype=object),
        lc_col=np.asarray(lc_col, dtype=object),
        time_col=np.asarray(time_col, dtype=object),
        target_col=np.asarray(target_col, dtype=object),
        seq_len=np.asarray(seq_len, dtype=np.int64),
        num_lc_classes=np.asarray(num_lc_classes, dtype=np.int64),
        lc_embed_dim=np.asarray(lc_embed_dim, dtype=np.int64),
        time_feature_dim=np.asarray(time_feature_dim, dtype=np.int64),
        max_gap_hours=np.asarray(-1 if max_gap_hours is None else max_gap_hours, dtype=np.float32),
        max_span_hours=np.asarray(-1 if max_span_hours is None else max_span_hours, dtype=np.float32),
        dt_clip_hours=np.asarray(dt_clip_hours, dtype=np.float32)
    )

    config = {
        "seq_len": int(seq_len),
        "time_col": time_col,
        "target_col": target_col,
        "forcing_cols": list(forcing_cols),
        "state_cols": list(state_cols),
        "static_cols": list(static_cols),
        "lc_col": lc_col,
        "num_forcing_features": len(forcing_cols),
        "num_state_features": len(state_cols),
        "num_static_features": len(static_cols),
        "num_lc_classes": int(num_lc_classes),
        "lc_embed_dim": int(lc_embed_dim),
        "time_feature_dim": int(time_feature_dim),
        "max_gap_hours": max_gap_hours,
        "max_span_hours": max_span_hours,
        "dt_clip_hours": dt_clip_hours,
        "scaler_file": "global_scalers.npz",
        "checkpoint_best_file": "checkpoint_best.pth",
        "checkpoint_best_full_file": "checkpoint_best_full.pth",
        "train_file_count": len(train_files),
        "val_file_count": len(val_files),
        "test_file_count": len(test_files),
        "train_sites": list(train_sites),
        "val_sites": list(val_sites),
        "test_sites": list(test_sites),
        "note": (
            "Global inversion must use exactly the same variable names, order, units, "
            "grid-time alignment, training scalers, irregular-time feature logic, "
            "max_gap_hours, max_span_hours, and dt_clip_hours."
        )
    }

    with open(config_json_path, "w", encoding="utf-8") as f:
        json.dump(
            config,
            f,
            ensure_ascii=False,
            indent=2
        )

    print("\n💾 已保存全球反演所需训练参数：")
    print(f"   - 标准化参数: {scaler_npz_path}")
    print(f"   - 输入配置:   {config_json_path}")

    return scaler_npz_path, config_json_path


# ============================================================
# 6. 主训练流程
# ============================================================

def train_time_aware_model(config):
    seq_len = config["seq_len"]
    batch_size = config["batch_size"]
    num_epochs = config["num_epochs"]
    learning_rate = config["learning_rate"]
    patience = config["patience"]

    time_col = config["time_col"]
    target_col = config["target_col"]

    num_lc_classes = config["num_lc_classes"]
    lc_embed_dim = config["lc_embed_dim"]
    time_feature_dim = config.get("time_feature_dim", 6)

    max_gap_hours = config.get("max_gap_hours", 6)
    max_span_hours = config.get("max_span_hours", 168)
    dt_clip_hours = config.get("dt_clip_hours", 240)

    data_folder = config["data_folder"]
    output_subfolder = config["output_subfolder"]
    output_img_folder = os.path.join(data_folder, output_subfolder)

    resume_training = config["resume_training"]

    forcing_cols = config["forcing_cols"]
    state_cols = config["state_cols"]
    static_cols = config["static_cols"]
    lc_col = config["lc_col"]

    train_sites = config["train_sites"]
    val_sites = config["val_sites"]
    test_sites = config["test_sites"]

    os.makedirs(output_img_folder, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("\n🖥️ 当前设备:", device)
    print("\n⏱️ 不等距时间序列参数")
    print(f"   time_feature_dim: {time_feature_dim}")
    print(f"   max_gap_hours:    {max_gap_hours}")
    print(f"   max_span_hours:   {max_span_hours}")
    print(f"   dt_clip_hours:    {dt_clip_hours}")

    check_site_overlap(
        train_sites=train_sites,
        val_sites=val_sites,
        test_sites=test_sites
    )

    all_files = glob.glob(
        os.path.join(data_folder, "*.[cC][sS][vV]")
    )

    if not all_files:
        print("❌ 错误：未在指定目录找到 CSV 文件。")
        return

    train_files, val_files, test_files = split_files_by_manual_sites(
        all_files=all_files,
        train_sites=train_sites,
        val_sites=val_sites,
        test_sites=test_sites
    )

    print("\n📌 最终站点划分")
    print(f"   手动指定训练站点数: {len(train_sites)}")
    print(f"   手动指定验证站点数: {len(val_sites)}")
    print(f"   手动指定测试站点数: {len(test_sites)}")

    print("\n📌 最终文件划分")
    print(f"   训练集文件: {len(train_files)}")
    print(f"   验证集文件: {len(val_files)}")
    print(f"   测试集文件: {len(test_files)}")

    print("\n🚉 实际参与训练的文件:")
    for f in train_files:
        print(f"   - {os.path.basename(f)}")

    train_dataset = TimeAwareMultiStationDataset(
        train_files,
        seq_len,
        target_col=target_col,
        time_col=time_col,
        forcing_cols=forcing_cols,
        state_cols=state_cols,
        static_cols=static_cols,
        lc_col=lc_col,
        split_type='train',
        max_gap_hours=max_gap_hours,
        max_span_hours=max_span_hours,
        dt_clip_hours=dt_clip_hours
    )

    f_mean_f = train_dataset.feat_mean_f
    f_std_f = train_dataset.feat_std_f

    f_mean_s = train_dataset.feat_mean_s
    f_std_s = train_dataset.feat_std_s

    s_mean = train_dataset.static_mean
    s_std = train_dataset.static_std

    t_mean = train_dataset.target_mean
    t_std = train_dataset.target_std

    save_training_artifacts(
        output_img_folder=output_img_folder,
        seq_len=seq_len,
        time_col=time_col,
        target_col=target_col,
        forcing_cols=forcing_cols,
        state_cols=state_cols,
        static_cols=static_cols,
        lc_col=lc_col,
        num_lc_classes=num_lc_classes,
        lc_embed_dim=lc_embed_dim,
        time_feature_dim=time_feature_dim,
        max_gap_hours=max_gap_hours,
        max_span_hours=max_span_hours,
        dt_clip_hours=dt_clip_hours,
        train_files=train_files,
        val_files=val_files,
        test_files=test_files,
        train_sites=train_sites,
        val_sites=val_sites,
        test_sites=test_sites,
        f_mean_f=f_mean_f,
        f_std_f=f_std_f,
        f_mean_s=f_mean_s,
        f_std_s=f_std_s,
        s_mean=s_mean,
        s_std=s_std,
        t_mean=t_mean,
        t_std=t_std
    )

    val_dataset = TimeAwareMultiStationDataset(
        val_files,
        seq_len,
        target_col=target_col,
        time_col=time_col,
        forcing_cols=forcing_cols,
        state_cols=state_cols,
        static_cols=static_cols,
        lc_col=lc_col,
        feat_mean_f=f_mean_f,
        feat_std_f=f_std_f,
        feat_mean_s=f_mean_s,
        feat_std_s=f_std_s,
        static_mean=s_mean,
        static_std=s_std,
        target_mean=t_mean,
        target_std=t_std,
        split_type='val',
        max_gap_hours=max_gap_hours,
        max_span_hours=max_span_hours,
        dt_clip_hours=dt_clip_hours
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model = TCN_Transformer_CrossAttention(
        num_forcing_features=len(forcing_cols),
        num_state_features=len(state_cols),
        seq_len=seq_len,
        num_static=len(static_cols),
        num_lc_classes=num_lc_classes,
        lc_embed_dim=lc_embed_dim,
        time_feature_dim=time_feature_dim
    ).to(device)

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    checkpoint_latest_path = os.path.join(
        output_img_folder,
        "checkpoint_latest.pth"
    )

    checkpoint_best_path = os.path.join(
        output_img_folder,
        "checkpoint_best.pth"
    )

    checkpoint_best_full_path = os.path.join(
        output_img_folder,
        "checkpoint_best_full.pth"
    )

    start_epoch = 0
    best_val_loss = float('inf')
    epochs_no_improve = 0

    if resume_training and os.path.exists(checkpoint_latest_path):
        print("\n🔄 检测到本地进度，尝试恢复...")

        try:
            checkpoint = torch.load(
                checkpoint_latest_path,
                map_location=device
            )

            model.load_state_dict(
                checkpoint['model_state_dict']
            )

            optimizer.load_state_dict(
                checkpoint['optimizer_state_dict']
            )

            start_epoch = checkpoint['epoch'] + 1
            best_val_loss = checkpoint['best_val_loss']
            epochs_no_improve = checkpoint['epochs_no_improve']

            print(f"✅ 从第 {start_epoch} 轮恢复，历史最佳 MSE: {best_val_loss:.4f}")

        except Exception as e:
            print(f"⚠️ 无法加载历史进度，将重新开始。错误信息: {e}")

    if start_epoch < num_epochs and epochs_no_improve < patience:
        print("\n🚀 开始训练...")
        print("-" * 40)

        for epoch in range(start_epoch, num_epochs):
            model.train()
            train_loss = 0.0

            for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, _ in train_loader:
                batch_forcing = batch_forcing.to(device)
                batch_state = batch_state.to(device)
                batch_time = batch_time.to(device)
                batch_static = batch_static.to(device)
                batch_lc = batch_lc.to(device)
                batch_y = batch_y.to(device)

                optimizer.zero_grad()

                outputs = model(
                    batch_forcing,
                    batch_state,
                    batch_time,
                    batch_static,
                    batch_lc
                )

                loss = criterion(
                    outputs,
                    batch_y
                )

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

                optimizer.step()

                train_loss += loss.item()

            model.eval()
            val_loss = 0.0

            with torch.no_grad():
                for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, _ in val_loader:
                    batch_forcing = batch_forcing.to(device)
                    batch_state = batch_state.to(device)
                    batch_time = batch_time.to(device)
                    batch_static = batch_static.to(device)
                    batch_lc = batch_lc.to(device)
                    batch_y = batch_y.to(device)

                    outputs = model(
                        batch_forcing,
                        batch_state,
                        batch_time,
                        batch_static,
                        batch_lc
                    )

                    loss = criterion(
                        outputs,
                        batch_y
                    )

                    val_loss += loss.item()

            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)

            print(
                f"Epoch [{epoch + 1:03d}/{num_epochs}] | "
                f"Train MSE (Std Scaled): {avg_train_loss:.4f} | "
                f"Val MSE (Std Scaled): {avg_val_loss:.4f}"
            )

            if avg_val_loss < best_val_loss:
                print(f"   🌟 新的最佳模型！MSE: {avg_val_loss:.4f}")

                best_val_loss = avg_val_loss
                epochs_no_improve = 0

                torch.save(
                    model.state_dict(),
                    checkpoint_best_path
                )

                torch.save({
                    'model_state_dict': model.state_dict(),
                    'seq_len': seq_len,
                    'time_col': time_col,
                    'target_col': target_col,
                    'forcing_cols': forcing_cols,
                    'state_cols': state_cols,
                    'static_cols': static_cols,
                    'lc_col': lc_col,
                    'num_lc_classes': num_lc_classes,
                    'lc_embed_dim': lc_embed_dim,
                    'time_feature_dim': time_feature_dim,
                    'max_gap_hours': max_gap_hours,
                    'max_span_hours': max_span_hours,
                    'dt_clip_hours': dt_clip_hours,
                    'feat_mean_f': f_mean_f,
                    'feat_std_f': f_std_f,
                    'feat_mean_s': f_mean_s,
                    'feat_std_s': f_std_s,
                    'static_mean': s_mean,
                    'static_std': s_std,
                    'target_mean': t_mean,
                    'target_std': t_std,
                    'train_sites': train_sites,
                    'val_sites': val_sites,
                    'test_sites': test_sites
                }, checkpoint_best_full_path)

            else:
                epochs_no_improve += 1
                print(f"   ⏳ 验证集未改善，早停计数: {epochs_no_improve}/{patience}")

            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
                'epochs_no_improve': epochs_no_improve,
                'seq_len': seq_len,
                'time_col': time_col,
                'target_col': target_col,
                'forcing_cols': forcing_cols,
                'state_cols': state_cols,
                'static_cols': static_cols,
                'lc_col': lc_col,
                'num_lc_classes': num_lc_classes,
                'lc_embed_dim': lc_embed_dim,
                'time_feature_dim': time_feature_dim,
                'max_gap_hours': max_gap_hours,
                'max_span_hours': max_span_hours,
                'dt_clip_hours': dt_clip_hours,
                'feat_mean_f': f_mean_f,
                'feat_std_f': f_std_f,
                'feat_mean_s': f_mean_s,
                'feat_std_s': f_std_s,
                'static_mean': s_mean,
                'static_std': s_std,
                'target_mean': t_mean,
                'target_std': t_std,
                'train_sites': train_sites,
                'val_sites': val_sites,
                'test_sites': test_sites
            }, checkpoint_latest_path)

            if epochs_no_improve >= patience:
                print("\n🛑 早停机制触发，训练结束。")
                break

            print("-" * 40)

    # ========================================================
    # 测试集评估
    # ========================================================

    if os.path.exists(checkpoint_best_path):
        print("\n🎯 开始测试集评估...")

        model.load_state_dict(
            torch.load(
                checkpoint_best_path,
                map_location=device
            )
        )

    model.eval()

    global_all_preds = []
    global_all_targets = []

    for test_file in test_files:
        station_name = os.path.splitext(os.path.basename(test_file))[0]

        try:
            single_test_dataset = TimeAwareMultiStationDataset(
                [test_file],
                seq_len,
                target_col=target_col,
                time_col=time_col,
                forcing_cols=forcing_cols,
                state_cols=state_cols,
                static_cols=static_cols,
                lc_col=lc_col,
                feat_mean_f=f_mean_f,
                feat_std_f=f_std_f,
                feat_mean_s=f_mean_s,
                feat_std_s=f_std_s,
                static_mean=s_mean,
                static_std=s_std,
                target_mean=t_mean,
                target_std=t_std,
                split_type='test',
                max_gap_hours=max_gap_hours,
                max_span_hours=max_span_hours,
                dt_clip_hours=dt_clip_hours
            )

        except Exception as e:
            print(f"⚠️ 测试站点 {station_name} 无法形成有效测试样本，跳过。原因: {e}")
            continue

        if len(single_test_dataset) == 0:
            continue

        single_test_loader = DataLoader(
            single_test_dataset,
            batch_size=batch_size,
            shuffle=False
        )

        all_preds = []
        all_targets = []
        all_times = []

        with torch.no_grad():
            for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, batch_dt in single_test_loader:
                batch_forcing = batch_forcing.to(device)
                batch_state = batch_state.to(device)
                batch_time = batch_time.to(device)
                batch_static = batch_static.to(device)
                batch_lc = batch_lc.to(device)

                outputs = model(
                    batch_forcing,
                    batch_state,
                    batch_time,
                    batch_static,
                    batch_lc
                )

                all_preds.extend(outputs.cpu().numpy())
                all_targets.extend(batch_y.numpy())
                all_times.extend(batch_dt)

        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)
        all_times = pd.to_datetime(all_times)

        all_preds = all_preds * t_std + t_mean
        all_targets = all_targets * t_std + t_mean

        valid_mask = all_targets >= 0

        plot_targets = all_targets[valid_mask]
        plot_preds = all_preds[valid_mask]
        plot_times = all_times[valid_mask]

        if len(plot_targets) == 0:
            continue

        global_all_preds.extend(plot_preds)
        global_all_targets.extend(plot_targets)

        station_mse = np.mean((plot_preds - plot_targets) ** 2)
        station_r2 = r2_score(plot_targets, plot_preds)

        print(
            f"📢 站点: {station_name} | "
            f"测试集 MSE: {station_mse:.4f} | "
            f"测试集 R²: {station_r2:.4f}"
        )

        window_size = 12

        all_targets_smooth = pd.Series(plot_targets).rolling(
            window=window_size,
            min_periods=1
        ).mean().values

        all_preds_smooth = pd.Series(plot_preds).rolling(
            window=window_size,
            min_periods=1
        ).mean().values

        # 图 1：滑动平均趋势图
        plt.figure(figsize=(15, 6))

        plt.plot(
            plot_times,
            plot_targets,
            color='royalblue',
            linewidth=0.5,
            alpha=0.2,
            label='Actual (Raw)'
        )

        plt.plot(
            plot_times,
            plot_preds,
            color='crimson',
            linewidth=0.5,
            alpha=0.2,
            label='Predicted (Raw)'
        )

        plt.plot(
            plot_times,
            all_targets_smooth,
            label=f'Actual GPP (MA-{window_size})',
            color='royalblue',
            linewidth=1.5,
            alpha=0.9
        )

        plt.plot(
            plot_times,
            all_preds_smooth,
            label=f'Predicted GPP (MA-{window_size})',
            color='crimson',
            linewidth=1.5,
            linestyle='--',
            alpha=0.9
        )

        plt.title(
            f'[{station_name}] Test Set GPP Prediction (Moving Average Window={window_size})',
            fontsize=14,
            fontname='Arial'
        )

        plt.xlabel('Time', fontsize=12, fontname='Arial')
        plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
        plt.legend(prop={'family': 'Arial'})

        ax = plt.gca()
        ax.tick_params(direction='in')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.grid(True, which='both', linestyle='--', alpha=0.5)
        plt.tight_layout()

        plt.savefig(
            os.path.join(output_img_folder, f"{station_name}_trend_ma.png"),
            dpi=300,
            facecolor='white',
            transparent=False
        )

        plt.close()

        # 图 2：30 天局部放大图
        zoom_days = 30
        steps_per_day = 24
        zoom_steps = zoom_days * steps_per_day
        zoom_steps = min(zoom_steps, len(plot_times))

        if zoom_steps > 0:
            peak_idx = np.argmax(all_targets_smooth)

            start_idx = max(
                0,
                peak_idx - zoom_steps // 2
            )

            end_idx = min(
                len(plot_times),
                start_idx + zoom_steps
            )

            if end_idx - start_idx < zoom_steps:
                start_idx = max(
                    0,
                    end_idx - zoom_steps
                )

            plt.figure(figsize=(15, 5))

            plt.plot(
                plot_times[start_idx:end_idx],
                plot_targets[start_idx:end_idx],
                label='Actual GPP (Raw)',
                color='royalblue',
                linewidth=1.5
            )

            plt.plot(
                plot_times[start_idx:end_idx],
                plot_preds[start_idx:end_idx],
                label='Predicted GPP (Raw)',
                color='crimson',
                linewidth=1.5,
                linestyle='--',
                alpha=0.8
            )

            peak_date_str = pd.to_datetime(
                plot_times[peak_idx]
            ).strftime('%Y-%m-%d')

            plt.title(
                f'[{station_name}] 30-Day Zoomed Prediction around {peak_date_str}',
                fontsize=14,
                fontname='Arial'
            )

            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'})

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

            plt.grid(True, which='both', linestyle='--', alpha=0.6)
            plt.xticks(rotation=30)
            plt.tight_layout()

            plt.savefig(
                os.path.join(output_img_folder, f"{station_name}_zoom_30days.png"),
                dpi=300,
                facecolor='white',
                transparent=False
            )

            plt.close()

        # 图 3：散点图
        plt.figure(figsize=(6, 6))

        plt.scatter(
            plot_targets,
            plot_preds,
            alpha=0.6,
            color='teal',
            s=15,
            edgecolors='k',
            linewidth=0.2
        )

        min_val = min(
            plot_targets.min(),
            plot_preds.min()
        )

        max_val = max(
            plot_targets.max(),
            plot_preds.max()
        )

        plt.plot(
            [min_val, max_val],
            [min_val, max_val],
            'r--',
            lw=2,
            label='1:1 Line'
        )

        plt.title(
            f'[{station_name}] Actual vs Predicted Scatter',
            fontname='Arial'
        )

        plt.xlabel('Actual GPP', fontname='Arial')
        plt.ylabel('Predicted GPP', fontname='Arial')
        plt.legend(prop={'family': 'Arial'})

        ax = plt.gca()
        ax.tick_params(direction='in')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()

        plt.savefig(
            os.path.join(output_img_folder, f"{station_name}_scatter.png"),
            dpi=300,
            facecolor='white',
            transparent=False
        )

        plt.close()

        # 图 4：样本最多的一年
        years = pd.Series(plot_times).dt.year
        unique_years = years.unique()

        if len(unique_years) > 0:
            target_year = years.value_counts().idxmax()
            year_mask = (years == target_year).values

            y_times = plot_times[year_mask]
            y_targets = plot_targets[year_mask]
            y_preds = plot_preds[year_mask]

            plt.figure(figsize=(15, 5))

            plt.plot(
                y_times,
                y_targets,
                label=f'Actual GPP ({target_year})',
                color='royalblue',
                linewidth=1.2,
                alpha=0.8
            )

            plt.plot(
                y_times,
                y_preds,
                label=f'Predicted GPP ({target_year})',
                color='crimson',
                linewidth=1.2,
                linestyle='--',
                alpha=0.8
            )

            plt.title(
                f'[{station_name}] Full Year GPP Time Series ({target_year})',
                fontsize=14,
                fontname='Arial'
            )

            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'}, loc='upper right')

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

            plt.grid(True, which='both', linestyle='--', alpha=0.4)
            plt.tight_layout()

            plt.savefig(
                os.path.join(output_img_folder, f"{station_name}_singleyear_{target_year}.png"),
                dpi=300,
                facecolor='white',
                transparent=False
            )

            plt.close()

    # ========================================================
    # 全局测试指标 + SHAP
    # ========================================================

    if len(global_all_targets) > 0:
        global_all_preds = np.array(global_all_preds)
        global_all_targets = np.array(global_all_targets)

        global_mse = np.mean((global_all_preds - global_all_targets) ** 2)
        global_r2 = r2_score(global_all_targets, global_all_preds)

        print("\n" + "=" * 50)
        print("🌎 所有站点测试集全局评估结果")
        print("=" * 50)
        print(f"有效总测试样本数，剔除 GPP < 0: {len(global_all_targets)}")
        print(f"Global Test MSE: {global_mse:.4f}")
        print(f"Global Test R² Score: {global_r2:.4f}")
        print("=" * 50)

        try:
            perform_shap_analysis(
                model,
                val_loader,
                device,
                output_img_folder,
                forcing_cols,
                state_cols
            )

        except Exception as e:
            print(f"⚠️ SHAP 分析生成失败: {e}")

    else:
        print("\n⚠️ 测试集没有得到有效样本，无法计算全局测试指标。")


# ============================================================
# 7. 你以后只需要修改这里
# ============================================================

USER_CONFIG = {
    # =========================
    # 路径设置
    # =========================
    "data_folder": r"D:\实验六\全波段全变量DT",
    "output_subfolder": "手动指定训练验证测试集_不等距时间_TCN",

    # =========================
    # 是否继续训练
    # 改了训练/验证/测试划分、时间特征维度、时间窗口逻辑后，建议保持 False
    # =========================
    "resume_training": False,

    # =========================
    # 训练参数
    # =========================
    "seq_len": 96,
    "batch_size": 64,
    "num_epochs": 100,
    "learning_rate": 0.001,
    "patience": 10,

    # =========================
    # 不等距时间序列参数
    # =========================
    # time_feature_dim 固定为 6：
    # sin(hour), cos(hour), sin(dayofyear), cos(dayofyear),
    # log1p(dt_prev_hours), log1p(age_to_target_hours)
    "time_feature_dim": 6,

    # 窗口内部相邻两条 EPIC 观测记录允许的最大间隔。
    # 如果有效样本太少，可以改成 8、12 或 None。
    "max_gap_hours": 6,

    # 一个 96 步窗口允许的最大总时间跨度。
    # 如果有效样本太少，可以改成 192、240 或 None。
    "max_span_hours": 168,

    # 时间间隔特征的最大截断值，防止极端断点影响模型。
    "dt_clip_hours": 240,

    # =========================
    # 字段名设置
    # =========================
    "time_col": "date",
    "target_col": "GPP_DT_VUT_REF",
    "lc_col": "Veg_ID",

    # =========================
    # 土地覆盖嵌入设置
    # 注意：Veg_ID 必须在 [0, num_lc_classes - 1] 范围内。
    # 如果你的 Veg_ID 是 1-13，需要先改成 0-12，或者把 num_lc_classes 改成 14。
    # =========================
    "num_lc_classes": 13,
    "lc_embed_dim": 8,

    # =========================
    # forcing 变量
    # =========================
    "forcing_cols": [
        'SW_IN_F',
        'SW_IN_POT',
        'CO2_F_MDS',
        'P_F',
        'VPD_F',
        'TA_F',
        'TS_F_MDS_1',
        'SWC_F_MDS_1',
        'WS_F'
    ],

    # =========================
    # state 变量
    # =========================
    "state_cols": [
        'EPIC_Available_Mask',
        'Band317nm_Ref',
        'Band325nm_Ref',
        'Band340nm_Ref',
        'Band388nm_Ref',
        'Band443nm_Ref',
        'Band551nm_Ref',
        'Band680nm_Ref',
        'Band688nm_Ref',
        'Band764nm_Ref',
        'Band780nm_Ref',
        'Mean_SZA',
        'Mean_VZA',
        'Mean_RAA'
    ],

    # =========================
    # static 变量
    # =========================
    "static_cols": [
        'Lat',
        'Long'
    ],

    # =========================
    # 训练集站点
    # =========================
    "train_sites": [
        'AU-Cpr_WSA',
        'AU-Cum_EBF',
        'AU-Fle_WSA',
        'AU-Gin_EBF',
        'AU-Rgf_CVM',
        'AU-Ya2_GRA',
        'BE-Vie_MF',
        'CA-Mer_WET',
        'CH-Lae_MF',
        'CH-Tnk_CRO',
        'CL-SDP_WET',
        'CN-Ash_EBF',
        'CN-BeO_MF',
        'CN-Dmn_CRO',
        'CN-HeM_DBF',
        'CN-Sdq_OSH',
        'CN-YaS_OSH',
        'CR-Fsc_CRO',
        'CZ-LnG_GRA',
        'CZ-Lnz_DBF',
        'CZ-Stn_DBF',
        'DE-Gri_GRA',
        'DE-GsB_GRA',
        'DE-Hai_DBF',
        'DE-RuC_OSH',
        'ES-Abr_SAV',
        'ES-Hen_ENF',
        'ES-LMa_SAV',
        'FI-Ken_ENF',
        'FI-Sii_WET',
        'FR-Fon_DBF',
        'FR-Tou_GRA',
        'JP-Tef_DNF',
        'KR-HcM_MF',
        'KR-Hnm_CRO',
        'KR-PcD_DBF',
        'KR-WdE_EBF',
        'NL-Loo_ENF',
        'NL-Vkp_GRA',
        'NZ-Lau_GRA',
        'SE-Deg_WET',
        'US-A39_CRO',
        'US-Bi2_CRO',
        'US-CdM_WSA',
        'US-ICh_OSH',
        'US-ICs_WET',
        'US-MOz_DBF',
        'US-NC2_ENF',
        'US-Ne1_CRO',
        'US-Rms_CSH',
        'US-Rpf_DBF',
        'US-Rws_OSH',
        'US-Ton_WSA',
        'US-UC2_CVM',
        'US-xDJ_ENF',
        'US-xDS_CVM',
        'US-xJE_ENF',
        'US-xMB_OSH',
        'US-xSB_ENF',
        'US-xSE_DBF',
        'US-xSJ_SAV',
        'US-xSL_CRO',
        'US-xWD_GRA',
        'US-xWR_ENF'
    ],

    # =========================
    # 验证集站点
    # =========================
    "val_sites": [
        'DE-RuM_CRO',
        'US-Mo1_CRO',
        'CN-Yuc_CRO',
        'CZ-KrP_CRO',
        'AU-MvB_CRO',
        'DE-RuS_CRO',
        'US-Ne2_CRO',
        'US-Ro1_CRO',
        'US-TKs_CSH',
        'US-UC1_CVM',
        'US-CMW_DBF',
        'US-xBL_DBF',
        'US-UMd_DBF',
        'IT-BsB_DBF',
        'US-MMS_DBF',
        'AU-Sno_EBF',
        'CA-TP3_ENF',
        'RU-Fyo_ENF',
        'US-CRK_ENF',
        'RU-Fy2_ENF',
        'DE-Har_ENF',
        'CA-TP4_ENF',
        'KR-ScC_ENF',
        'US-SP1_ENF',
        'ES-HeB_ENF',
        'IT-SR2_ENF',
        'IL-Yat_ENF',
        'CH-Crs_GRA',
        'US-SRG_GRA',
        'FR-Mej_GRA',
        'DE-RuR_GRA',
        'US-Var_GRA',
        'AU-Sil_GRA',
        'US-xAE_GRA',
        'US-Wkg_GRA',
        'US-xCL_GRA',
        'FR-CLt_GRA',
        'CH-Fru_GRA',
        'US-xUN_MF',
        'US-xJR_OSH',
        'ES-LM2_SAV',
        'DE-Amv_WET',
        'US-SRM_WSA'
    ],

    # =========================
    # 测试集站点
    # =========================
    "test_sites": [
        'CH-Oe2_CRO',
        'FR-Aur_CRO',
        'CN-Jzh_CRO',
        'US-Ne3_CRO',
        'KR-CRK_CRO',
        'CN-Nmn_CRO',
        'BE-Lon_CRO',
        'US-Rls_CSH',
        'US-A37_CVM',
        'DK-Sor_DBF',
        'US-xBR_DBF',
        'US-xUK_DBF',
        'US-xTR_DBF',
        'JP-Fhk_DNF',
        'AU-GWW_EBF',
        'SE-Ros_ENF',
        'FI-Var_ENF',
        'KR-AdC_ENF',
        'US-Me6_ENF',
        'DE-Obe_ENF',
        'IT-Lav_ENF',
        'BE-Bra_ENF',
        'EE-Stg_ENF',
        'FI-Hyy_ENF',
        'CZ-RAJ_ENF',
        'US-MEF_ENF',
        'FR-Nzn_GRA',
        'US-ONA_GRA',
        'BE-Dor_GRA',
        'AU-Wel_GRA',
        'US-xDC_GRA',
        'AU-Ya1_GRA',
        'US-Kon_GRA',
        'IT-Tor_GRA',
        'US-xKZ_GRA',
        'US-KFS_GRA',
        'AU-Sam_GRA',
        'KR-JjM_MF',
        'CN-Ha2_OSH',
        'ES-LM1_SAV',
        'JP-BBY_WET',
        'US-SRS_WSA'
    ]
}


if __name__ == "__main__":
    train_time_aware_model(USER_CONFIG)

D:\miniconda3\envs\dl_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🖥️ 当前设备: cuda

⏱️ 不等距时间序列参数
   time_feature_dim: 6
   max_gap_hours:    6
   max_span_hours:   168
   dt_clip_hours:    240

📁 文件划分完成
   文件夹内 CSV 总数: 530
   训练集文件数: 64
   验证集文件数: 43
   测试集文件数: 42
   未使用 CSV 文件数: 381

📌 最终站点划分
   手动指定训练站点数: 64
   手动指定验证站点数: 43
   手动指定测试站点数: 42

📌 最终文件划分
   训练集文件: 64
   验证集文件: 43
   测试集文件: 42

🚉 实际参与训练的文件:
   - AU-Cpr_WSA_Merged_EPIC_Time.csv
   - AU-Cum_EBF_Merged_EPIC_Time.csv
   - AU-Fle_WSA_Merged_EPIC_Time.csv
   - AU-Gin_EBF_Merged_EPIC_Time.csv
   - AU-Rgf_CVM_Merged_EPIC_Time.csv
   - AU-Ya2_GRA_Merged_EPIC_Time.csv
   - BE-Vie_MF_Merged_EPIC_Time.csv
   - CA-Mer_WET_Merged_EPIC_Time.csv
   - CH-Lae_MF_Merged_EPIC_Time.csv
   - CH-Tnk_CRO_Merged_EPIC_Time.csv
   - CL-SDP_WET_Merged_EPIC_Time.csv
   - CN-Ash_EBF_Merged_EPIC_Time.csv
   - CN-BeO_MF_Merged_EPIC_Time.csv
   - CN-Dmn_CRO_Merged_EPIC_Time.csv
   - CN-HeM_DBF_Merged_EPIC_Time.csv
   - CN-Sdq_OSH_Merged_EPIC_Time.csv
   - CN-YaS_OSH_Merged_EPIC_Time.csv
   - CR-Fsc_CRO_Merged_EPIC_Time.

C:\Users\admin\AppData\Local\Temp\ipykernel_28368\2419043533.py:814: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(
C:\Users\admin\AppData\Local\Temp\ipykernel_28368\2419043533.py:838: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


✅ SHAP 生成完成，保存至: D:\实验六\全波段全变量DT\手动指定训练验证测试集_不等距时间_TCN
